# 05 — Retrieval: candidate products

Four sources, fit on **train only**, scored on warm validation users: item-KNN (`cf`), TF-IDF (`tfidf`), MiniLM embeddings (`emb`), trending popularity (`pop`). Products the user already bought in train are excluded everywhere.

In [1]:
import sys
import time

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

sys.path.insert(0, "../src")
from shoprank.config import load_config
from shoprank.retrieval.baselines import PopularRecommender
from shoprank.retrieval.candidates import run_retrieval
from shoprank.retrieval.embeddings import load_or_encode
from shoprank.retrieval.generators import ItemKNN, PopularityScorer, ProfileScorer
from shoprank.retrieval.matrix import interaction_matrix, make_index

cfg = load_config("../config.yaml")
R = cfg["retrieval"]
T1, PROC, MODELS = cfg.t1, cfg.path("processed"), cfg.path("models")

train = pd.read_parquet(PROC / "train.parquet")
val = pd.read_parquet(PROC / "val.parquet")
items = pd.read_parquet(PROC / "items.parquet")
eval_users = pd.read_parquet(PROC / "eval_users.parquet")

## 1 — User × product sparse matrix

Columns follow `items.parquet` (all 9,999 products) so every source scores the same columns; products first seen after T1 are empty columns in `X` but still reachable by the text sources.

In [2]:
user_index = make_index(train["user_id"])
item_index = make_index(items["parent_asin"])
user_ids = np.array(list(user_index))
item_ids = np.array(list(item_index))

X = interaction_matrix(train, user_index, item_index)
X_liked = interaction_matrix(train[train["liked"] == 1], user_index, item_index)
no_liked = X_liked.getnnz(axis=1) == 0
X_profile = (X_liked + X.multiply(no_liked[:, None])).tocsr()

density = X.nnz / (X.shape[0] * X.shape[1])
print(f"X: {X.shape[0]:,} users x {X.shape[1]:,} products, {X.nnz:,} filled ({density:.4%}), "
      f"{(X.data.nbytes + X.indices.nbytes + X.indptr.nbytes) / 1e6:.1f} MB sparse vs {X.shape[0] * X.shape[1] * 4 / 1e9:.2f} GB dense")
print(f"users with no liked train product (profile uses all their products): {no_liked.sum():,}")

X: 22,324 users x 9,999 products, 149,524 filled (0.0670%), 1.3 MB sparse vs 0.89 GB dense
users with no liked train product (profile uses all their products): 464


## 2 — Item-KNN collaborative filtering

In [3]:
t = time.time()
cf = ItemKNN(n_neighbors=R["item_knn_neighbors"]).fit(X)
print(f"fit in {time.time() - t:.1f}s, similarity matrix nnz {cf.S.nnz:,}")

titles = items["title"].str[:55].to_numpy()
probe = item_index[train["parent_asin"].value_counts().index[0]]
row = cf.S[probe].toarray().ravel()
print("most bought:", titles[probe])
for j in np.argsort(-row)[:5]:
    print(f"  {row[j]:.3f}  {titles[j]}")

fit in 0.3s, similarity matrix nnz 425,950
most bought: amFilm Tempered Glass Screen Protector for Nintendo Swi
  0.154  Orzly Carrying case for Nintendo Switch OLED and Switch
  0.142  Nintendo Switch Pro Controller
  0.136  Mumba Case for Nintendo Switch, [Heavy Duty] Slim Rubbe
  0.131  The Legend of Zelda: Breath of the Wild Master Edition 
  0.108  Mario Kart 8 Deluxe – Booster Course Pass - Nintendo Sw


## 3 — TF-IDF (fit on products seen in train, transform all)

In [4]:
seen_in_train = items["parent_asin"].isin(train["parent_asin"]).to_numpy()
tfidf_vec = TfidfVectorizer(**dict(R["tfidf"], ngram_range=tuple(R["tfidf"]["ngram_range"])))
tfidf_vec.fit(items.loc[seen_in_train, "item_text"])
V_tfidf = tfidf_vec.transform(items["item_text"])
tfidf = ProfileScorer(V_tfidf, X_profile)
print(f"fit on {seen_in_train.sum():,} train products, vocabulary {len(tfidf_vec.vocabulary_):,}, matrix {V_tfidf.shape}")

fit on 8,519 train products, vocabulary 50,000, matrix (9999, 50000)


## 4 — Sentence embeddings (`all-MiniLM-L6-v2`), cached to `models/item_embeddings.npy`

In [5]:
t = time.time()
E = load_or_encode(item_ids, items["item_text"].tolist(), MODELS / "item_embeddings.npy",
                   R["embedding_model"], R["embedding_batch_size"])
print(f"embeddings {E.shape} in {time.time() - t:.0f}s")
emb = ProfileScorer(E, X_profile)

embeddings (9999, 384) in 0s


## 5 — Popularity (trending, last 30 days before T1)

In [6]:
trending = PopularRecommender(train, days=R["trending_days"], cutoff=T1)
pop = PopularityScorer(trending.ranking, item_index)
sources = {"cf": cf, "tfidf": tfidf, "emb": emb, "pop": pop}

## 6 / 7 — Combine and measure Recall@100 on validation

First a quick run on 10% of warm users, then all of them.

Equal quotas under-use the strongest source: 25 each (~91 products) scores below trending@100 alone. Popularity gets a larger quota. `balanced` (~95 products) is the like-for-like comparison with the single sources at 100; `saved` (~150) is what goes to the ranker.

In [7]:
truth = interaction_matrix(val[val["liked"] == 1], user_index, item_index)
warm_users = eval_users.query("split == 'val' and group == 'warm'")["user_id"]
warm_rows = warm_users.map(user_index).to_numpy()

rng = np.random.default_rng(42)
sample_rows = rng.choice(warm_rows, size=len(warm_rows) // 10, replace=False)

unions = {
    "union equal 25 each": {"cf": 25, "tfidf": 25, "emb": 25, "pop": 25},
    "union balanced 20/20/10/50": {"cf": 20, "tfidf": 20, "emb": 10, "pop": 50},
    "union saved 30/30/30/75": {"cf": 30, "tfidf": 30, "emb": 30, "pop": 75},
    "union equal 50 each": {"cf": 50, "tfidf": 50, "emb": 50, "pop": 50},
}
SAVED = "union saved 30/30/30/75"

runs = {}
for name, rows in [("10% sample", sample_rows), ("all warm users", warm_rows)]:
    t = time.time()
    runs[name], candidates = run_retrieval(sources, rows, X, truth, user_ids, item_ids,
                                           unions=unions, keep=SAVED, eval_n=100)
    print(f"{name}: {len(rows):,} users in {time.time() - t:.1f}s")

pd.concat(runs, axis=1).round(4)

10% sample: 454 users in 0.7s


all warm users: 4,549 users in 7.2s


10% sample           all warm users          
                               recall  avg size         recall  avg size
cf@100                         0.0519  100.0000         0.0575  100.0000
tfidf@100                      0.0913  100.0000         0.0770  100.0000
emb@100                        0.0644  100.0000         0.0626  100.0000
pop@100                        0.1270  100.0000         0.1028  100.0000
union equal 25 each            0.1117   91.1344         0.1007   90.7344
union balanced 20/20/10/50     0.1377   94.6300         0.1112   94.5612
union saved 30/30/30/75        0.1767  153.5088         0.1477  153.0149
union equal 50 each            0.1787  180.2930         0.1484  179.4212

In [8]:
full = runs["all warm users"]["recall"]
single = [f"{s}@100" for s in sources]
best_single = full[single].max()
for label in ["union balanced 20/20/10/50", SAVED]:
    assert full[label] > best_single, f"{label} should beat every single source"
print(f"best single source: {full[single].idxmax()} {best_single:.4f}   "
      f"balanced union (~95): {full['union balanced 20/20/10/50']:.4f}   saved union (~150): {full[SAVED]:.4f}")

found = candidates[candidates["label"] == 1]
print(f"liked products in candidates: {len(found):,} of {truth[warm_rows].nnz:,}")

quotas = unions[SAVED]
in_quota = {s: (candidates.groupby("user_id")[f"{s}_score"].rank(ascending=False, method="first") <= n)[found.index].to_numpy()
            for s, n in quotas.items()}
only = {s: int((in_quota[s] & ~np.logical_or.reduce([in_quota[o] for o in sources if o != s])).sum()) for s in sources}
print("liked products in the saved candidates found ONLY by this source:", only)

best single source: pop@100 0.1028   balanced union (~95): 0.1112   saved union (~150): 0.1477
liked products in candidates: 1,417 of 10,774


liked products in the saved candidates found ONLY by this source: {'cf': 169, 'tfidf': 157, 'emb': 102, 'pop': 744}


## Save `candidates_val.parquet`

One row per (user, candidate): `cf_score`, `tfidf_score`, `emb_score`, `pop_rank` (1 = most trending), and `label` = liked in validation (for the Phase 7 ranker).

In [9]:
candidates["pop_rank"] = pop.rank[candidates["parent_asin"].map(item_index).to_numpy()]
candidates = candidates.drop(columns="pop_score")
candidates.to_parquet(PROC / "candidates_val.parquet", index=False)

per_user = candidates.groupby("user_id").size()
print(f"{len(candidates):,} rows, {per_user.size:,} users, candidates per user: "
      f"mean {per_user.mean():.0f}, min {per_user.min()}, max {per_user.max()}, positive rate {candidates['label'].mean():.2%}")
candidates.head()

696,065 rows, 4,549 users, candidates per user: mean 153, min 124, max 165, positive rate 0.20%


,user_id,parent_asin,cf_score,tfidf_score,emb_score,label,pop_rank
0,AGBP6HAY2K6QPNW3KENGNXURANWQ,B00X8XIN7S,0.105409,0.048261,0.617894,0,3762
1,AGBP6HAY2K6QPNW3KENGNXURANWQ,B09DFCB66S,0.000000,0.043983,0.498405,0,65
2,AGBP6HAY2K6QPNW3KENGNXURANWQ,B09M2CPHDL,0.000000,0.047291,0.491133,0,61
3,AGBP6HAY2K6QPNW3KENGNXURANWQ,B0BTM9LJM4,0.000000,0.038983,0.406231,0,13
4,AGBP6HAY2K6QPNW3KENGNXURANWQ,B08NDYPWK7,0.191835,0.058208,0.312062,0,7462
